<a href="https://colab.research.google.com/github/LS-SEC/ls-report-template/blob/main/%EC%9D%80%ED%96%89_%EC%9D%BC%EC%9D%BC%EB%8D%B0%EC%9D%B4%ED%84%B0_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q --upgrade pykrx tabulate yfinance

In [4]:
import os
from importlib.metadata import version
from google.colab import userdata

os.environ['KRX_ID'] = userdata.get('KRX_ID')   # 금고에서 아이디 꺼내기
os.environ['KRX_PW'] = userdata.get('KRX_PW')   # 금고에서 비번 꺼내기

from pykrx import stock   # ← 반드시 자격증명 설정 '후에' import

_t = stock.get_index_ohlcv("20240102", "20240105", "1001")
print(f"pykrx {version('pykrx')}")
print("✓ KRX 로그인 정상" if not _t.empty else "⚠ 로그인 실패 — Secret 확인")


KRX 로그인 시도...
  로그인 ID: mujige01
KRX 로그인 완료.
  로그인 시간: 2026-07-06 09:35:20
  만료 시간: 2026-07-06 10:35:20
pykrx 1.2.8
✓ KRX 로그인 정상


In [5]:
STOCKS = {
    '105560': 'KB금융', '055550': '신한지주', '086790': '하나금융지주',
    '316140': '우리금융지주', '024110': '기업은행', '138930': 'BNK금융지주',
    '139130': 'iM금융지주', '175330': 'JB금융지주', '323410': '카카오뱅크',
}

In [7]:
# ================================================================
# [셀 4 · v2.1] 데이터 추출 — 시계열 + 수급심화 + 밸류에이션 + 대외지표
#
#  전제: 셀1(설치)·셀2(로그인)·셀3(STOCKS) 실행 후 이 셀 실행
#  셀1:  !pip install -q --upgrade pykrx tabulate yfinance
#
#  v2.1 변경: ① 지수·수급 한 표 통합 ② 관심종목 한 표 통합(1W 제외)
#             ③ 대외 표 가로형(높이 축소) ④ 대외지표 '범위 가드' 내장
#                (값이 정상 범위를 벗어나면 N/A + 경고 — 단위 오류·티커 변경 자동 검출)
#  ⚠️ 첫 실행 [CHECK] 3곳: A 밸류에이션 함수 / B 종목별 수급 / C 대외 표 값
# ================================================================
import datetime as _dt
from zoneinfo import ZoneInfo
from html import escape
import pandas as pd
from IPython.display import HTML, display

KST = ZoneInfo('Asia/Seoul')
INDICES = {'1001': '코스피', '2001': '코스닥'}
SECTORS = {'5044':'반도체','5043':'자동차','5045':'헬스케어','5046':'은행',
           '5048':'에너지화학','5049':'철강','5051':'방송통신','5052':'건설',
           '5054':'증권','5055':'기계장비','5056':'보험','5057':'운송',
           '5061':'경기소비재','5062':'필수소비재','5063':'K콘텐츠',
           '5064':'정보기술','5065':'유틸리티'}

# ---------- 1) 기준일 (16시 이후=당일 확정치, 이전=직전 영업일) ----------
_now = _dt.datetime.now(KST)
_e = _now.strftime('%Y%m%d')
_probe = [d.strftime('%Y%m%d') for d in stock.get_index_ohlcv((_now - _dt.timedelta(days=14)).strftime('%Y%m%d'), _e, '1001').index]
if _now.hour < 16 and _probe and _probe[-1] == _e:
    _probe = _probe[:-1]
BASE = _probe[-1]
BASE_F = f'{BASE[:4]}-{BASE[4:6]}-{BASE[6:]}'
YEAR = int(BASE[:4])
_FROM = f'{YEAR-1}1201'          # 전년 12월부터 조회 → YTD 기준(전년 마지막 종가) 확보
print('기준일:', BASE_F)

# ---------- 2) 수익률 계산기 ----------
def _rets(close, d1_override=None):
    last = close.iloc[-1]
    def pct(n):
        return (last / close.iloc[-1 - n] - 1) * 100 if len(close) > n else None
    d1 = d1_override if d1_override is not None else pct(1)
    w1, m1 = pct(5), pct(21)                    # 1W=5영업일, 1M=21영업일
    prev = close[close.index.year < YEAR]
    ytd = (last / prev.iloc[-1] - 1) * 100 if len(prev) else None
    return last, d1, w1, m1, ytd

def _f(x, nd=2, comma=False):
    if x is None: return 'N/A'
    return f'{x:,.{nd}f}' if comma else f'{x:.{nd}f}'

# ---------- 3) 수급 (시장) + 수급상세 ----------
print('수급 수집...')
flow_by_mkt, detail_rows = {}, []
for mkt, name in [('KOSPI','코스피'), ('KOSDAQ','코스닥')]:
    inv = stock.get_market_trading_value_by_investor(BASE, BASE, mkt)['순매수']
    flow_by_mkt[name] = [f"{inv['개인']/1e9:+,.1f}",
                         f"{(inv['외국인']+inv['기타외국인'])/1e9:+,.1f}",
                         f"{inv['기관합계']/1e9:+,.1f}"]
    if mkt == 'KOSPI':   # 기관 세부 — 해석 서술 전용 (보고서 표 미반영)
        for k in ['금융투자','투신','보험','연기금']:
            if k in inv.index:
                detail_rows.append([k, f'{inv[k]/1e9:+,.1f}'])
detail_md = pd.DataFrame(detail_rows, columns=['주체(코스피)','순매수(십억)']).to_markdown(index=False)

# ---------- 4) 지수·수급 통합 표 + 업종 (시계열) ----------
print('지수·업종 수집 (연초부터, 30~60초)...')
def _ts(code):
    c = stock.get_index_ohlcv(_FROM, BASE, code)['종가']
    last, d1, w1, m1, ytd = _rets(c)
    return [_f(last, 2, True), _f(d1), _f(w1), _f(m1), _f(ytd)]

idx_rows = [[n] + _ts(c) + flow_by_mkt[n] for c, n in INDICES.items()]
idx_md = pd.DataFrame(idx_rows, columns=['구분','종가','1D(%)','1W(%)','1M(%)','YTD(%)','개인(십억)','외국인(십억)','기관(십억)']).to_markdown(index=False)

sec_rows = [[n] + _ts(c) for c, n in SECTORS.items()]
sec_rows.sort(key=lambda r: float(r[2]), reverse=True)
sec_md = pd.DataFrame(sec_rows, columns=['구분','종가','1D(%)','1W(%)','1M(%)','YTD(%)']).to_markdown(index=False)

# ---------- 5) 관심종목 통합 표 (가격·수익률 + 수급 + 밸류, 1W 제외) ----------
print('관심종목 수집...')
# [CHECK A] 밸류에이션 — 오류 시 대안: stock.get_market_fundamental_by_ticker(BASE, market="ALL")
try:
    _fund = stock.get_market_fundamental(BASE, market='ALL')
except Exception:
    _fund = stock.get_market_fundamental_by_ticker(BASE, market='ALL')

def _stk_flow(code):
    # [CHECK B] 종목 단위 수급 — 오류 시 N/A로 진행
    try:
        inv = stock.get_market_trading_value_by_investor(BASE, BASE, code)['순매수']
        return (inv.get('외국인', 0) + inv.get('기타외국인', 0)) / 1e8, inv.get('기관합계', 0) / 1e8   # 억원
    except Exception:
        return None, None

stk_rows = []
for code, name in STOCKS.items():
    df = stock.get_market_ohlcv(_FROM, BASE, code)
    d1_official = df['등락률'].iloc[-1] if '등락률' in df.columns else None
    last, d1, w1, m1, ytd = _rets(df['종가'], d1_override=d1_official)
    frn, ins = _stk_flow(code)
    per = _fund.loc[code, 'PER'] if code in _fund.index else None
    pbr = _fund.loc[code, 'PBR'] if code in _fund.index else None
    dvd = _fund.loc[code, 'DIV'] if code in _fund.index else None
    stk_rows.append([name, f'{int(last):,}', _f(d1), _f(m1), _f(ytd),
                     f'{frn:+,.0f}' if frn is not None else 'N/A',
                     f'{ins:+,.0f}' if ins is not None else 'N/A',
                     _f(per), _f(pbr), _f(dvd)])
stk_rows.sort(key=lambda r: float(r[2]) if r[2] != 'N/A' else -999, reverse=True)   # 1D 내림차순
stk_md = pd.DataFrame(stk_rows, columns=['종목','종가','1D(%)','1M(%)','YTD(%)','외국인(억)','기관(억)','PER(배)','PBR(배)','배당(%)']).to_markdown(index=False)

# ---------- 6) 대외지표 (yfinance) — 가로형 + 범위 가드 ----------
print('대외지표 수집...')
ext_close, ext_chg, ext_names, ext_date = [], [], [], ''
try:
    import yfinance as yf
    # (티커, 표기명, 정상 범위) — 범위 밖 값 = 단위 오류·티커 변경 의심 → N/A + 경고
    EXT = [('^GSPC','S&P500',(2000,20000)), ('^IXIC','나스닥',(5000,60000)),
           ('^SOX','SOX',(1000,20000)), ('KRW=X','달러/원',(800,2500)),
           ('CL=F','WTI',(10,300)), ('^TNX','美10년물',(0.3,12))]
    for tk, name, (lo, hi) in EXT:
        ext_names.append(name)
        try:
            h = yf.Ticker(tk).history(period='15d')['Close'].dropna()
            h = h[[d.strftime('%Y%m%d') < BASE for d in h.index]]   # 기준일 '간밤'까지의 세션만
            last, prev = float(h.iloc[-1]), float(h.iloc[-2])
            if tk == '^TNX':   # [CHECK C] ^TNX는 수익률×10 제공 → /10 보정
                last, prev = last/10, prev/10
            if not (lo <= last <= hi) or not (lo <= prev <= hi):
                print(f'⚠ {name} 값 검증 실패({last:.2f}) — 정상범위({lo}~{hi}) 밖, N/A 처리')
                ext_close.append('N/A'); ext_chg.append('N/A'); continue
            if tk == '^TNX':
                ext_close.append(f'{last:.2f}'); ext_chg.append(f'{last-prev:+.2f}%p')
            else:
                ext_close.append(f'{last:,.2f}'); ext_chg.append(f'{(last/prev-1)*100:+.2f}')
            ext_date = h.index[-1].strftime('%m/%d')
        except Exception:
            ext_close.append('N/A'); ext_chg.append('N/A')
except Exception as e:
    print('yfinance 실패(대외 표 생략 가능):', e)
if ext_names:
    ext_md = pd.DataFrame([['종가'] + ext_close, ['등락'] + ext_chg],
                          columns=['항목'] + ext_names).to_markdown(index=False)
else:
    ext_md = '(수집 실패)'

# ---------- 7) 조립 + [전체 복사] ----------
report = (f"## 대외 — 간밤 미 증시 (현지 {ext_date} 종가 · 美10년물 등락은 %p · SOX=필라델피아반도체)\n{ext_md}\n\n"
          f"## 지수·수급 (기준일 {BASE_F} · 수급: 순매수, 십억원)\n{idx_md}\n\n"
          f"## 수급상세 (코스피 기관 분해 — 서술용, 표 재출력 금지)\n{detail_md}\n\n"
          f"## 업종 (KRX 섹터지수)\n{sec_md}\n\n"
          f"## 관심종목 (수급: 당일 순매수 억원 / PER·PBR·배당: 트레일링 — 12M FWD 아님)\n{stk_md}")
display(HTML(
    "<button style='padding:8px 18px;font-size:14px;margin:8px 0;cursor:pointer;' "
    "onclick=\"navigator.clipboard.writeText(document.getElementById('rpt').value)"
    ".then(()=>this.textContent='복사 완료!')\">[전체 복사]</button>"
    f"<textarea id='rpt' rows='30' style='width:100%;font-family:monospace;font-size:12px;'>{escape(report)}</textarea>"
))
print('버튼이 안 눌리면: 상자 클릭 → Ctrl+A → Ctrl+C')

기준일: 2026-07-06
수급 수집...
지수·업종 수집 (연초부터, 30~60초)...
관심종목 수집...
대외지표 수집...


버튼이 안 눌리면: 상자 클릭 → Ctrl+A → Ctrl+C
